# Correcciones para `PF_v1_paraStreamlit-2.ipynb`

Este notebook contiene **solo las celdas que hay que corregir o reemplazar**.
No modifica el resto del notebook — solo sustituye las celdas problemáticas.

---

## Diagnóstico completo de errores

| # | Celda | Problema | Impacto |
|---|---|---|---|
| 1 | 110 | `is_short_track` sobreescrito con lógica invertida (`> 3.0` en lugar de `< 2.5`) | El modelo aprende que "corta" = >3 min, al revés |
| 2 | 84 | `dropna()` sin `inplace=True` ni reasignación | No tiene efecto, df_clean no se modifica |
| 3 | 190 | `predecir_hit()` usa `le_genero`, `le_pais`, `genre_encoded`, `country_encoded`, `listeners_log`, `n_tracks_artista`, `peso_en_artista`, `engagement` | Todo eso no existe en este notebook → `NameError` |
| 4 | 202 | Guarda `le_genero` y `le_pais` que no están definidos | `NameError` |
| 5 | 209 | Guarda `le_genero`, `le_pais` (no existen) + `rf_reg` (259 MB, excede límite GitHub) | `NameError` + fallo en `git push` |

**Raíz de todos los errores:** las celdas 190-211 son código de una versión anterior del notebook donde el encoder de género se llamaba `le_genero` y usaba la columna `genre_tag`. En esta versión el encoder correcto es `le_tag` y usa la columna `tag`.

**Lo que NO hay que tocar:** el bloque de entrenamiento (celdas 173-187) está bien. `le_tag`, `FEATURES`, `rf_clf` y `rf_reg` son coherentes entre sí.

---
## CORRECCIÓN 1 — Celda 110: `is_short_track` (reemplazar la celda completa)

**Problema:** la celda 110 sobreescribe `is_short_track` con `duration_min > 3.0` (canciones LARGAS),
contradiciendo la celda 109 que lo define correctamente como `< 2.5`.
El modelo se entrena con la definición de la celda 110 → las "canciones cortas" en el modelo son en realidad las largas.

**Solución:** eliminar la celda 110. La celda 109 es la correcta.

In [ ]:
# CELDA 109 — CORRECTA (mantener tal cual)
# is_short_track: 1 si la canción dura menos de 2.5 min (formato TikTok/Reels)
df_clean['is_short_track'] = (df_clean['duration_min'] < 2.5).astype(int)
print(f'Canciones cortas (<2.5 min): {df_clean["is_short_track"].sum():,} ({df_clean["is_short_track"].mean()*100:.1f}%)')

# CELDA 110 — ELIMINAR (o comentar). Sobreescribe con lógica invertida.
# df_clean['is_short_track'] = (df_clean['duration_min'] > 3.0).astype(int)  ← BORRAR

---
## CORRECCIÓN 2 — Celda 84: `dropna()` sin efecto (reemplazar)

**Problema:** `df_clean.dropna(subset=['name', 'artist'])` no modifica `df_clean`
porque falta `inplace=True` o la reasignación.

In [ ]:
# CELDA 84 — CORRECCIÓN: reasignar el resultado
antes = len(df_clean)
df_clean = df_clean.dropna(subset=['name', 'artist'])
print(f'Filas eliminadas por name/artist nulo: {antes - len(df_clean):,}')
print(f'Filas restantes: {len(df_clean):,}')

---
## CORRECCIÓN 3 — Celda 190: `predecir_hit()` para Streamlit (reemplazar)

**Problema:** la función usa variables que no existen en este notebook:
- `le_genero` → no definido (el encoder correcto es `le_tag`, celda 173)
- `le_pais` → no definido (no hay encoding de país en este notebook)
- `listeners_log` → no existe (se llama `log_listeners`)
- `genre_encoded` → no existe (se llama `tag_encoded`)
- `country_encoded` → no existe en FEATURES
- `n_tracks_artista` → no existe (se llama `artist_track_count`)
- `peso_en_artista` → no existe (se llama `track_share_of_artist`)
- `engagement` → no existe en FEATURES

**Solución:** reemplazar por la misma función que ya funciona en celda 187, añadiendo solo el parámetro `pais` como informativo (sin encoding, porque el modelo no lo usa).

In [ ]:
# CELDA 190 — CORRECCIÓN COMPLETA
# Replica la lógica de celda 187 (que SÍ es coherente con el modelo entrenado).
# Usa le_tag (definido en celda 173) y los nombres de FEATURES correctos.

def predecir_hit(nombre, artista, duracion_min, genero, oyentes_estimados, pais=None):
    """
    Predice la probabilidad de que una canción sea un hit.

    Parámetros:
        nombre            — nombre de la canción
        artista           — nombre del artista
        duracion_min      — duración en minutos (ej: 3.5)
        genero            — género musical. Debe estar en le_tag.classes_
                            (los géneros son los tags de Last.fm, ej: 'rock', 'pop')
        oyentes_estimados — estimación de oyentes únicos
        pais              — informativo, no afecta la predicción (el modelo no tiene país)

    Devuelve: probabilidad de hit (0-100%)
    """
    # Encoding de género con le_tag — el mismo encoder usado en el entrenamiento
    genero_enc = le_tag.transform([genero])[0] if genero in le_tag.classes_ else 0

    # Vector de features — mismo orden y nombres que FEATURES del entrenamiento
    datos = pd.DataFrame([{
        'log_listeners'          : np.log1p(oyentes_estimados),
        'duration_min'           : duracion_min,
        'is_short_track'         : int(duracion_min < 2.5),
        'tag_encoded'            : genero_enc,
        'artist_track_count'     : 1,    # artista nuevo: 1 track
        'track_share_of_artist'  : 1.0,  # único track del artista
        'playcount_per_listener' : 5.0,  # engagement inicial estimado
    }])
    datos = datos[FEATURES]  # mismo orden que durante el entrenamiento

    probabilidad = rf_clf.predict_proba(datos)[0][1] * 100

    if probabilidad >= 70:
        clasificacion = '🚀 Hit potencial'
    elif probabilidad >= 45:
        clasificacion = '🟡 Potencial medio'
    else:
        clasificacion = '📉 Bajo potencial'

    print('=' * 50)
    print(f'  🎵 {nombre} — {artista}')
    print('=' * 50)
    print(f'  Probabilidad de hit:  {probabilidad:.1f}%')
    print(f'  Clasificación:        {clasificacion}')
    print(f'  Género:               {genero}')
    if pais:
        print(f'  País:                 {pais}  (informativo)')
    print(f'  Duración:             {duracion_min:.1f} min', end='')
    print(f' (corta ⏱️)' if duracion_min < 2.5 else '')
    print('=' * 50)

    return probabilidad


# Prueba
predecir_hit(
    nombre='Mi Canción',
    artista='Mi Artista',
    duracion_min=2.3,
    genero='pop',
    oyentes_estimados=50000,
    pais='spain'
)

---
## CORRECCIÓN 4 — Celdas 202 y 209: guardado de modelos (reemplazar ambas)

**Problemas:**
- Guardan `le_genero` y `le_pais` que no existen → `NameError`
- `modelo_plays_reg.pkl` pesa 259 MB → bloquea `git push` (límite GitHub: 100 MB)

**Solución:**
- Guardar `le_tag` (el encoder que sí existe y se usó en el entrenamiento)
- Usar `joblib` con compresión para el regresor
- Excluir el regresor de Git con `.gitignore`

In [ ]:
# CELDAS 202 + 209 — REEMPLAZAR por este bloque único

import joblib, os

# Verificación previa: confirmar coherencia entre modelo y encoder
assert hasattr(rf_clf, 'n_features_in_'), 'rf_clf no está entrenado'
assert rf_clf.n_features_in_ == len(FEATURES), (
    f'ERROR: rf_clf espera {rf_clf.n_features_in_} features '
    f'pero FEATURES tiene {len(FEATURES)}. '
    f'Vuelve a ejecutar las celdas 173-183.'
)
print('✅ Verificación OK: modelo y FEATURES son coherentes')
print(f'   Features: {FEATURES}')
print(f'   Géneros disponibles ({len(le_tag.classes_)}): {list(le_tag.classes_[:8])}...')
print()

os.makedirs('models', exist_ok=True)

# Clasificador: pocos MB, se puede subir a GitHub
joblib.dump(rf_clf,  'models/modelo_hits_clf.pkl')
print(f'modelo_hits_clf.pkl  → {os.path.getsize("models/modelo_hits_clf.pkl")/1e6:.1f} MB  ✅ OK para GitHub')

# Regresor: ~260 MB. Se guarda con compresión pero SE EXCLUYE de Git (.gitignore)
joblib.dump(rf_reg,  'models/modelo_plays_reg.pkl', compress=3)
print(f'modelo_plays_reg.pkl → {os.path.getsize("models/modelo_plays_reg.pkl")/1e6:.1f} MB  ⚠️  excluir de Git')

# Encoder de género — le_tag (NO le_genero, que no existe en este notebook)
joblib.dump(le_tag,  'models/le_tag.pkl')
print(f'le_tag.pkl           → {os.path.getsize("models/le_tag.pkl")/1e3:.1f} KB  ✅ OK para GitHub')

# Lista de features — el orden es crítico para la predicción
with open('models/features.txt', 'w') as f:
    f.write('\n'.join(FEATURES))
print(f'features.txt         → guardado')

print()
print('✅ Guardado correcto.')
print()
print('⚠️  Añade esto a tu .gitignore para evitar el error de push:')
print('   models/modelo_plays_reg.pkl')
print('   data/raw/*.csv')
print('   data/processed/*.csv')

In [ ]:
# TEST FINAL: simular exactamente cómo Streamlit cargará el modelo
import joblib

rf_clf_test   = joblib.load('models/modelo_hits_clf.pkl')
le_tag_test   = joblib.load('models/le_tag.pkl')
with open('models/features.txt') as f:
    feats_test = [l.strip() for l in f if l.strip()]

# Predicción de prueba
genero_test = 'rock'
enc_test = le_tag_test.transform([genero_test])[0] if genero_test in le_tag_test.classes_ else 0

datos_test = pd.DataFrame([{
    'log_listeners'          : np.log1p(50000),
    'duration_min'           : 3.2,
    'is_short_track'         : 0,
    'tag_encoded'            : enc_test,
    'artist_track_count'     : 1,
    'track_share_of_artist'  : 1.0,
    'playcount_per_listener' : 5.0,
}])[feats_test]

prob_test = rf_clf_test.predict_proba(datos_test)[0][1] * 100
print(f'✅ Test de carga OK: probabilidad de hit = {prob_test:.1f}%')
print(f'   Features cargadas: {feats_test}')
print(f'   Géneros disponibles: {len(le_tag_test.classes_)}')

---
## Error de Git: archivo demasiado grande

Ejecuta estos comandos en la terminal del Codespace para resolver el bloqueo de push:

```bash
# 1. Quitar el archivo grande del seguimiento de Git (sin borrar el archivo)
git rm --cached src/models/modelo_plays_reg.pkl

# 2. Añadir al .gitignore (si no lo está ya)
echo 'models/modelo_plays_reg.pkl' >> .gitignore
echo 'src/models/modelo_plays_reg.pkl' >> .gitignore

# 3. Commit y push
git add .gitignore
git commit -m 'remove large model file, update .gitignore'
git push origin main
```

Si el archivo ya está en el historial de commits anteriores y sigue bloqueando:

```bash
# Eliminar del historial completo (cuidado: reescribe el historial)
git filter-branch --force --index-filter \
  'git rm --cached --ignore-unmatch src/models/modelo_plays_reg.pkl' \
  --prune-empty --tag-name-filter cat -- --all

git push origin main --force
```